# Baseline Model – Intrusion Detection (Supervised Classification)

## Goal
The goal of this notebook is to establish a **baseline supervised classification model** for the cybersecurity intrusion detection task.

Each sample represents a single network session or access event, and the objective is to predict whether the session corresponds to:
- **0** → Normal behavior
- **1** → Malicious activity (attack)

### Baseline Definition
- **Baseline model:** Random Forest Classifier
- **Problem type:** Binary supervised classification
- **Input:** Preprocessed and split data
  (`X_train, X_val, X_test, y_train, y_val, y_test`)
- **Output:**
  - Classification metrics (Accuracy, Precision, Recall, F1, ROC-AUC)
  - Confusion Matrix and ROC Curve
  - Saved metrics and plots for later comparison with main models

This baseline provides a reference point against which more advanced models (e.g., XGBoost) will be evaluated.


In [5]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

RANDOM_STATE = 42

# Output directories
FIG_DIR = "../results/charts/phase-1/baseline"
METRICS_PATH = "../results/metrics/phase-1/baseline_metrics.json"

# Create output directories if they do not exist
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(os.path.dirname(METRICS_PATH), exist_ok=True)

print("Baseline configuration loaded.")


Baseline configuration loaded.


## Data Loading Strategy
The final version of this notebook expects preprocessed and split data to be loaded using a shared function:
  - `load_processed_splits()`
- This function will provide:
  - `X_train, X_val, X_test`
  - `y_train, y_val, y_test`

In [8]:
print("Loading preprocessed train/test data...")

train_path = "../data/processed/cybersecurity_intrusion_train_preprocessed.csv"
test_path  = "../data/processed/cybersecurity_intrusion_test_preprocessed.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

X_train = train_df.drop(columns=["attack_detected"])
y_train = train_df["attack_detected"]

X_test  = test_df.drop(columns=["attack_detected"])
y_test  = test_df["attack_detected"]

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Loading preprocessed train/test data...
Train shape: (6103, 17)
Validation shape: (1526, 17)
Test shape: (1908, 17)


## Baseline Model Training – Random Forest

In this step, we train a **Random Forest classifier** as the baseline model for the intrusion detection task.

Random Forest is selected as the baseline because:
- It performs well on **tabular data** with mixed feature types
- It can model **non-linear relationships** without complex feature engineering
- It is robust to noise and outliers
- It requires minimal preprocessing compared to other models

The model is trained using the **training split** and evaluated on the **test split** using standard classification metrics.
A fixed random state is used to ensure **reproducibility** of the results.

**Validation Strategy**

The validation set is used only for **sanity checking** the model performance.
No hyperparameter tuning is performed at this stage, as the goal is to establish a baseline reference.
Final results are reported on the test set.


This baseline serves as a reference point for comparison with more advanced models (e.g., XGBoost) in later experiments.


In [9]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    class_weight="balanced"
)
rf.fit(X_train, y_train)
y_pred_val = rf.predict(X_val)
val_acc = rf.score(X_val, y_val)
y_pred_test = rf.predict(X_test)
test_acc = rf.score(X_test, y_test)
y_proba_test = rf.predict_proba(X_test)[:, 1]
print("Val accuracy:", val_acc)
print("Test accuracy:", test_acc)

Val accuracy: 0.8990825688073395
Test accuracy: 0.8841719077568134


## Confusion Matrix

In [10]:
from sklearn.metrics import confusion_matrix

#calculate
cm = confusion_matrix(y_test, y_pred_test)
#result
plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
plt.title("Confusion Matrix - Random Forest (Baseline)")
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ["Normal", "Attack"])
plt.yticks(tick_marks, ["Normal", "Attack"])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j, i, cm[i, j],
            horizontalalignment="center",
            color="white" if cm[i, j] > cm.max() / 2 else "black"
        )
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
#save
cm_path = os.path.join(FIG_DIR, "confusion_matrix.png")
plt.savefig(cm_path)
plt.close()

print(f"Confusion Matrix saved to {cm_path}")


Confusion Matrix saved to ../results/charts/baseline\confusion_matrix.png


## ROC-AUC

In [11]:
from sklearn.metrics import roc_curve, roc_auc_score

#calculate
fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
roc_auc = roc_auc_score(y_test, y_proba_test)
#result
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random guess")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Random Forest (Baseline)")
plt.legend(loc="lower right")
plt.grid(True)
#save
roc_path = os.path.join(FIG_DIR, "roc_curve.png")
plt.savefig(roc_path)
plt.close()

print(f"ROC Curve saved to {roc_path}")


ROC Curve saved to ../results/charts/baseline\roc_curve.png


## classification report

In [12]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_test))

              precision    recall  f1-score   support

           0       0.83      1.00      0.90      1055
           1       0.99      0.75      0.85       853

    accuracy                           0.88      1908
   macro avg       0.91      0.87      0.88      1908
weighted avg       0.90      0.88      0.88      1908



## Save metrics

In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import json
import os

test_accuracy = accuracy_score(y_test, y_pred_test)
test_precision = precision_score(y_test, y_pred_test)
test_recall = recall_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test)

metrics = {
    "model": "RandomForestClassifier",
    "split": "test",
    "params": {
        "n_estimators": 200,
        "random_state": RANDOM_STATE,
        "class_weight": "balanced"
    },
    "accuracy": float(test_accuracy),
    "precision": float(test_precision),
    "recall": float(test_recall),
    "f1": float(test_f1),
    "roc_auc": float(roc_auc),
    "confusion_matrix": cm.tolist()
}

metrics_path = "../results/metrics/phase-1/baseline_metrics.json"
os.makedirs(os.path.dirname(metrics_path), exist_ok=True)

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Baseline metrics saved to {metrics_path}")

Baseline metrics saved to ../results/metrics/baseline_metrics.json


## Final Baseline Analysis – Random Forest

The Random Forest baseline model achieves an overall accuracy of 0.88 on the test set, indicating solid general classification performance.

### Class-wise Performance

For the **Normal class (0)**:
- Recall = 1.00
  The model correctly identifies all normal sessions.
- Precision = 0.83
  Some attack samples are misclassified as normal.

For the **Attack class (1)**:
- Precision = 0.99
  When the model predicts an attack, it is almost always correct.
- Recall = 0.75
  However, 25% of malicious sessions are missed (false negatives).

### ROC Analysis

The model achieves an AUC of 0.878, demonstrating good discriminative ability between normal and malicious sessions.
The ROC curve lies significantly above the random baseline, confirming that the model captures meaningful attack-related patterns.

### Security-Oriented Interpretation

In intrusion detection systems (IDS), **false negatives (missed attacks)** are typically more critical than false positives.
While the model produces very few false alarms (high precision for attacks), the recall of 0.75 indicates that some attacks remain undetected.

This suggests that the baseline model is relatively conservative: it predicts attacks only when highly confident.

### Conclusion

The Random Forest baseline provides:
- Strong overall performance
- Very low false alarm rate
- Good but improvable attack detection capability

However, there is room for improvement in increasing recall for the attack class without significantly increasing false positives.

This motivates the use of more advanced models such as XGBoost in Phase 2, where boosting techniques may better capture complex patterns and improve detection performance.